# Dia 12 — itertools & functools: Pipelines de Dados Eficientes

**Autor:** Gutemberg Rapôso · [GitHub](https://github.com/Gutembergfox) · Palmas-TO, Brasil
**Fase 1 — Python Basics** · Dia 12/28 · 25/10/2026

## 1. Introdução: por que `itertools` e `functools` importam para Data Science

- **`itertools`** oferece blocos de construção para trabalhar com **iteradores preguiçosos (lazy)**:
  os elementos são produzidos sob demanda, um de cada vez, sem materializar listas inteiras na memória.
  Isso é decisivo ao processar corpora grandes, logs ou streams de dados.
- **`functools`** oferece ferramentas de **programação funcional**: fixar argumentos (`partial`),
  reduzir sequências (`reduce`), memorizar resultados (`lru_cache`) e gerar comparações (`total_ordering`).

> 💡 **Nota de aprendizado:** já estudei *generators* (Dia 4) e *decorators* (Dia 11). Hoje o foco é
> **compor** iteradores e funções prontas da biblioteca padrão — escritas em C, rápidas e testadas —
> em vez de reinventar laços manuais.

**Datasets reais usados:** `seaborn` (titanic, penguins), `sklearn.datasets` (wine, diabetes) e o
**Brown Corpus** do NLTK (corpus de inglês americano de 1961, ~1 milhão de palavras, 15 gêneros).

In [1]:
import sys
import timeit
import operator
import itertools as it
import functools as ft
from collections import Counter

import numpy as np
import pandas as pd
import seaborn as sns
import nltk
from sklearn.datasets import load_wine, load_diabetes

for pkg in ("brown", "stopwords"):
    nltk.download(pkg, quiet=True)

from nltk.corpus import brown, stopwords

print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")

Python 3.11.17 | pandas 2.3.3 | numpy 1.26.4


In [2]:
# Load the real datasets used throughout the notebook
titanic = sns.load_dataset("titanic")
penguins = sns.load_dataset("penguins").dropna()
wine = load_wine(as_frame=True)
diabetes = load_diabetes(as_frame=True)

print("titanic :", titanic.shape)
print("penguins:", penguins.shape)
print("wine    :", wine.frame.shape)
print("diabetes:", diabetes.frame.shape)
print("brown   :", len(brown.words()), "tokens |", len(brown.categories()), "categories")

titanic : (891, 15)
penguins: (333, 7)
wine    : (178, 14)
diabetes: (442, 11)


brown   : 1161192 tokens | 15 categories


## 2. `itertools` — Iteradores Infinitos: `count()`, `cycle()`, `repeat()`

Esses iteradores **nunca terminam sozinhos**. Por isso sempre os combinamos com algo que
limite o consumo: `zip()` (para no menor iterável), `islice()` ou um `break`.

| Função | O que produz | Uso típico em DS |
|---|---|---|
| `count(start, step)` | start, start+step, ... | IDs, índices, timestamps sintéticos |
| `cycle(iterable)` | repete o iterável em loop | atribuir *folds*, cores, turmas |
| `repeat(obj, n)` | o mesmo objeto n vezes (ou infinito) | argumentos constantes em `map` |

In [3]:
# count(): assign sequential IDs to the first sentences of the Brown "learned" genre
sentences = brown.sents(categories="learned")
for sent_id, sent in zip(it.count(start=1000, step=1), sentences[:4]):
    print(sent_id, " ".join(sent[:10]), "...")

1000 1 . ...
1001 Introduction ...
1002 It has recently become practical to use the radio emission ...
1003 The results of present observations of the thermal radio emission ...


In [4]:
# cycle(): round-robin assignment of Titanic passengers to 5 cross-validation folds
folds = it.cycle(range(5))
titanic_cv = titanic.assign(fold=[next(folds) for _ in range(len(titanic))])
print(titanic_cv["fold"].value_counts().sort_index())

fold
0    179
1    178
2    178
3    178
4    178
Name: count, dtype: int64


In [5]:
# repeat(): supply a constant argument to map() — round each wine feature to 2 decimals
alcohol = wine.frame["alcohol"].head(6).tolist()
rounded = list(map(round, alcohol, it.repeat(1)))
print("original:", alcohol)
print("rounded :", rounded)

original: [14.23, 13.2, 13.16, 14.37, 13.24, 14.2]
rounded : [14.2, 13.2, 13.2, 14.4, 13.2, 14.2]


> 💡 **Nota de aprendizado:** `cycle()` guarda internamente uma cópia dos elementos já vistos —
> por isso funciona até com geradores. Já `repeat(x, n)` é mais eficiente que `[x] * n` quando
> só precisamos **iterar**, pois não cria lista.

## 3. `itertools` — Iteradores Finitos

Agora as ferramentas de "encanamento" que mais uso em pipelines:
`chain`, `islice`, `takewhile`, `dropwhile`, `accumulate`, `zip_longest` e `groupby`.

### 3.1 `chain()` e `chain.from_iterable()`
Concatena iteráveis **sem criar uma lista nova** — ideal para juntar documentos de vários gêneros.

In [6]:
# chain(): merge tokens from two Brown genres into one lazy stream
news_tokens = brown.words(categories="news")
editorial_tokens = brown.words(categories="editorial")
merged = it.chain(news_tokens, editorial_tokens)
print(list(it.islice(merged, 12)))

# chain.from_iterable(): flatten a list of tokenized sentences into a single token stream
flat = it.chain.from_iterable(brown.sents(categories="humor")[:3])
print(list(flat))

['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', 'Friday', 'an', 'investigation', 'of', "Atlanta's", 'recent']
['It', 'was', 'among', 'these', 'that', 'Hinkle', 'identified', 'a', 'photograph', 'of', 'Barco', '!', '!', 'For', 'it', 'seems', 'that', 'Barco', ',', 'fancying', 'himself', 'a', "ladies'", 'man', '(', 'and', 'why', 'not', ',', 'after', 'seven', 'marriages', '?', '?', ')', ',', 'had', 'listed', 'himself', 'for', 'Mormon', 'Beard', 'roles', 'at', 'the', 'instigation', 'of', 'his', 'fourth', 'murder', 'victim', 'who', 'had', 'said', ':', '``', 'With', 'your', 'beard', ',', 'dear', ',', 'you', 'ought', 'to', 'be', 'in', 'movies', "''", '!', '!']


### 3.2 `islice()`
Fatiamento para **qualquer iterador** (inclusive infinitos ou geradores), com `start`, `stop`, `step`.
Diferente de `lista[a:b]`, não exige indexação nem carrega tudo na memória.

In [7]:
def read_lowercase_tokens(category: str):
    """Yield lowercase alphabetic tokens from a Brown corpus category (lazy)."""
    return (w.lower() for w in brown.words(categories=category) if w.isalpha())


# Take tokens 100..120 from a generator — impossible with normal slicing
print(list(it.islice(read_lowercase_tokens("fiction"), 100, 120)))
# Every 1000th token (a cheap systematic sample)
print(list(it.islice(read_lowercase_tokens("fiction"), 0, 10_000, 1000)))

['up', 'with', 'a', 'certain', 'amount', 'of', 'work', 'at', 'home', 'there', 'was', 'little', 'danger', 'of', 'his', 'losing', 'a', 'term', 'scotty', 'accepted']
['scotty', 'encounter', 'they', 'had', 'back', 'was', 'a', 'had', 'too', 'about']


### 3.3 `takewhile()` e `dropwhile()`
- `takewhile(pred, it)`: entrega elementos **enquanto** o predicado for verdadeiro e para no primeiro falso.
- `dropwhile(pred, it)`: **descarta** enquanto for verdadeiro e entrega todo o resto.

Atenção: os dois param de avaliar o predicado depois da primeira mudança — não são filtros!

In [8]:
# Diabetes dataset sorted by BMI: split patients below/above a BMI threshold (scaled feature)
bmi_sorted = sorted(diabetes.frame["bmi"])
threshold = 0.0  # features are mean-centered, so 0.0 is the average BMI
below = list(it.takewhile(lambda x: x < threshold, bmi_sorted))
above = list(it.dropwhile(lambda x: x < threshold, bmi_sorted))
print(f"below average: {len(below)} | at/above average: {len(above)} | total: {len(bmi_sorted)}")

# NLP: skip leading punctuation in a sentence, then keep everything else
sent = ["``", "''", "--", "The", "jury", "said", "it", "--", "did", "."]
print(list(it.dropwhile(lambda t: not t.isalpha(), sent)))

below average: 247 | at/above average: 195 | total: 442
['The', 'jury', 'said', 'it', '--', 'did', '.']


### 3.4 `accumulate()`
Gera **somas acumuladas** (ou qualquer operação binária: `max`, `operator.mul`...).
Útil para curvas de crescimento de vocabulário, totais corridos e máximos móveis.

In [9]:
# Running total of Titanic fares and running maximum
fares = titanic["fare"].head(8).round(2).tolist()
print("fares        :", fares)
print("running sum  :", [round(x, 2) for x in it.accumulate(fares)])
print("running max  :", list(it.accumulate(fares, max)))
# Survival probability chain as a product example (multiplicative accumulation)
print("cum. product :", list(it.accumulate([0.9, 0.8, 0.95], operator.mul)))

fares        : [7.25, 71.28, 7.92, 53.1, 8.05, 8.46, 51.86, 21.08]
running sum  : [7.25, 78.53, 86.45, 139.55, 147.6, 156.06, 207.92, 229.0]
running max  : [7.25, 71.28, 71.28, 71.28, 71.28, 71.28, 71.28, 71.28]
cum. product : [0.9, 0.7200000000000001, 0.684]


In [10]:
def vocabulary_growth(tokens, checkpoints):
    """Return the number of distinct types seen after each checkpoint (type-token curve).

    Uses accumulate() to carry a growing set of seen types through the stream.
    """
    seen_counts = it.accumulate(
        tokens,
        lambda acc, tok: acc | {tok},
        initial=frozenset(),
    )
    checkpoints = set(checkpoints)
    return {i: len(s) for i, s in enumerate(seen_counts) if i in checkpoints}


growth = vocabulary_growth(it.islice(read_lowercase_tokens("news"), 5000), range(0, 5001, 1000))
print(growth)

{0: 0, 1000: 445, 2000: 768, 3000: 1070, 4000: 1324, 5000: 1511}


> 💡 **Nota de aprendizado (Linguística):** a curva *types × tokens* acima ilustra a **Lei de Heaps** —
> o vocabulário cresce sublinearmente com o tamanho do texto. (A versão com `frozenset` é didática;
> para corpora grandes, um `set` mutável num laço é mais eficiente.)

### 3.5 `zip_longest()`
Como `zip()`, mas continua até o **maior** iterável, preenchendo com `fillvalue`.
Útil para alinhar sequências de tamanhos diferentes (ex.: frases paralelas, colunas desiguais).

In [11]:
# Align the first sentence of three Brown genres token by token
first = {cat: brown.sents(categories=cat)[0][:8] for cat in ("news", "romance", "science_fiction")}
for row in it.zip_longest(*first.values(), fillvalue="<pad>"):
    print(" | ".join(f"{t:<12}" for t in row))

The          | They         | Now         
Fulton       | neither      | that        
County       | liked        | he          
Grand        | nor          | knew        
Jury         | disliked     | himself     
said         | the          | to          
Friday       | Old          | be          
an           | Man          | self        


### 3.6 `groupby()`
Agrupa elementos **consecutivos** com a mesma chave. ⚠️ Os dados precisam estar **ordenados pela chave**,
senão grupos repetidos aparecem separados — erro clássico!

In [12]:
# groupby on Titanic records: survival rate by passenger class (data must be sorted by key)
records = sorted(
    titanic[["pclass", "survived"]].itertuples(index=False),
    key=operator.attrgetter("pclass"),
)
for pclass, group in it.groupby(records, key=operator.attrgetter("pclass")):
    survived = [r.survived for r in group]
    print(f"class {pclass}: n={len(survived):3d} | survival rate={np.mean(survived):.3f}")

# Cross-check with pandas
print(titanic.groupby("pclass")["survived"].mean().round(3).to_dict())

class 1: n=216 | survival rate=0.630
class 2: n=184 | survival rate=0.473
class 3: n=491 | survival rate=0.242
{1: 0.63, 2: 0.473, 3: 0.242}


In [13]:
# Pitfall demo: unsorted input produces fragmented groups
unsorted_keys = titanic["embark_town"].dropna().head(12).tolist()
print([(k, len(list(g))) for k, g in it.groupby(unsorted_keys)])
print([(k, len(list(g))) for k, g in it.groupby(sorted(unsorted_keys))])

[('Southampton', 1), ('Cherbourg', 1), ('Southampton', 3), ('Queenstown', 1), ('Southampton', 3), ('Cherbourg', 1), ('Southampton', 2)]
[('Cherbourg', 2), ('Queenstown', 1), ('Southampton', 9)]


In [14]:
# NLP use: group words by first letter, or by length — run-length style statistics
words = sorted({w.lower() for w in brown.words(categories="humor") if w.isalpha()}, key=len)
length_dist = {length: len(list(g)) for length, g in it.groupby(words, key=len)}
print("word-length distribution (types):", dict(it.islice(length_dist.items(), 12)))

word-length distribution (types): {1: 7, 2: 35, 3: 177, 4: 509, 5: 654, 6: 704, 7: 722, 8: 585, 9: 445, 10: 291, 11: 157, 12: 91}


## 4. `itertools` — Combinatória: `product()`, `permutations()`, `combinations()`

| Função | Ordem importa? | Repetição? | Exemplo com "ABC", r=2 |
|---|---|---|---|
| `product` | sim | sim | AA AB AC BA ... (9) |
| `permutations` | sim | não | AB AC BA BC CA CB (6) |
| `combinations` | não | não | AB AC BC (3) |
| `combinations_with_replacement` | não | sim | AA AB AC BB BC CC (6) |

Em DS, isso aparece em **grid search de hiperparâmetros**, **pares de features** para correlação
e **n-gramas / pares de palavras** em NLP.

In [15]:
# product(): a hyperparameter grid (what GridSearchCV does internally)
grid = {"max_depth": [3, 5, None], "min_samples_leaf": [1, 5], "criterion": ["gini", "entropy"]}
configs = [dict(zip(grid, values)) for values in it.product(*grid.values())]
print(f"{len(configs)} configurations, e.g.: {configs[:2]}")

12 configurations, e.g.: [{'max_depth': 3, 'min_samples_leaf': 1, 'criterion': 'gini'}, {'max_depth': 3, 'min_samples_leaf': 1, 'criterion': 'entropy'}]


In [16]:
# combinations(): find the most correlated feature pairs in the Wine dataset
features = wine.frame.columns.drop("target")
pairs = (
    (a, b, wine.frame[a].corr(wine.frame[b]))
    for a, b in it.combinations(features, 2)
)
top = sorted(pairs, key=lambda x: abs(x[2]), reverse=True)[:5]
print(f"{len(list(it.combinations(features, 2)))} pairs evaluated")
for a, b, r in top:
    print(f"{a:>30} ~ {b:<30} r={r:+.3f}")

78 pairs evaluated
                 total_phenols ~ flavanoids                     r=+0.865
                    flavanoids ~ od280/od315_of_diluted_wines   r=+0.787
                 total_phenols ~ od280/od315_of_diluted_wines   r=+0.700
                    flavanoids ~ proanthocyanins                r=+0.653
                       alcohol ~ proline                        r=+0.644


In [17]:
# permutations(): ordered word pairs — e.g., generating candidate word-order exercises
phrase = ["she", "has", "already", "left"]
orders = [" ".join(p) for p in it.permutations(phrase)]
print(len(orders), "orderings; first five:", orders[:5])

24 orderings; first five: ['she has already left', 'she has left already', 'she already has left', 'she already left has', 'she left has already']


In [18]:
# combinations() for NLP: co-occurrence of content words inside the same sentence
stop = set(stopwords.words("english"))

def content_words(sentence):
    """Return sorted unique lowercase alphabetic non-stopword tokens of a sentence."""
    return sorted({w.lower() for w in sentence if w.isalpha() and w.lower() not in stop})

cooc = Counter(
    it.chain.from_iterable(
        it.combinations(content_words(s), 2) for s in brown.sents(categories="learned")
    )
)
print(cooc.most_common(8))

[(('another', 'one'), 40), (('states', 'united'), 36), (('one', 'two'), 36), (('af', 'c'), 32), (('cent', 'per'), 29), (('dominant', 'stress'), 29), (('completion', 'onset'), 28), (('af', 'line'), 27)]


> 💡 **Nota de aprendizado:** o número de combinações explode rápido (C(n,2) = n(n−1)/2).
> Como `combinations` é *lazy*, consigo encadear com `Counter` sem materializar todos os pares
> numa lista intermediária.

## 5. `functools` — `partial()`, `reduce()`, `lru_cache()`, `total_ordering`

### 5.1 `partial()`
Cria uma nova função com **alguns argumentos já fixados**. Excelente para configurar funções
de pré-processamento uma vez e reutilizá-las em `map`, `apply` ou pipelines.

In [19]:
def clean_tokens(tokens, lowercase=True, remove_stopwords=False, min_len=1):
    """Normalize a token list: optional lowercasing, stopword removal and length filter."""
    out = (t.lower() if lowercase else t for t in tokens if t.isalpha())
    if remove_stopwords:
        out = (t for t in out if t.lower() not in stop)
    return [t for t in out if len(t) >= min_len]


# Two preconfigured cleaners built from the same function
strict_clean = ft.partial(clean_tokens, remove_stopwords=True, min_len=3)
light_clean = ft.partial(clean_tokens, lowercase=False)

sample = brown.sents(categories="editorial")[2]
print("raw   :", sample[:14])
print("light :", light_clean(sample)[:14])
print("strict:", strict_clean(sample)[:14])
print(strict_clean.func.__name__, strict_clean.keywords)

raw   : ['It', 'was', 'faced', 'immediately', 'with', 'a', 'showdown', 'on', 'the', 'schools', ',', 'an', 'issue', 'which']
light : ['It', 'was', 'faced', 'immediately', 'with', 'a', 'showdown', 'on', 'the', 'schools', 'an', 'issue', 'which', 'was']
strict: ['faced', 'immediately', 'showdown', 'schools', 'issue', 'met', 'squarely', 'conjunction', 'governor', 'decision', 'risk', 'abandoning', 'public', 'education']
clean_tokens {'remove_stopwords': True, 'min_len': 3}


In [20]:
# partial() with pandas: a reusable quantile-based binning function
def bin_column(series, q, labels):
    """Bin a numeric Series into quantile-based categories."""
    return pd.qcut(series, q=q, labels=labels)

tercile = ft.partial(bin_column, q=3, labels=["low", "mid", "high"])
penguins_binned = penguins.assign(mass_band=tercile(penguins["body_mass_g"]))
print(pd.crosstab(penguins_binned["species"], penguins_binned["mass_band"]))

mass_band  low  mid  high
species                  
Adelie      77   62     7
Chinstrap   36   31     1
Gentoo       0   17   102


### 5.2 `reduce()`
Aplica uma função binária **cumulativamente**, reduzindo a sequência a um único valor.
`accumulate()` mostra os passos intermediários; `reduce()` devolve só o resultado final.

In [21]:
# reduce(): merge several per-genre Counters into one global frequency table
genres = ["news", "editorial", "reviews"]
counters = [Counter(read_lowercase_tokens(g)) for g in genres]
total = ft.reduce(operator.add, counters)
print({g: sum(c.values()) for g, c in zip(genres, counters)}, "->", sum(total.values()))

# reduce(): intersect vocabularies — words shared by all three genres
shared = ft.reduce(set.intersection, (set(c) for c in counters))
print(f"shared vocabulary across {genres}: {len(shared)} types")

{'news': 83562, 'editorial': 52765, 'reviews': 34108} -> 170435
shared vocabulary across ['news', 'editorial', 'reviews']: 2788 types


In [22]:
# reduce() to compose functions: build a single text-normalization pipeline
def compose(*funcs):
    """Compose functions left-to-right: compose(f, g)(x) == g(f(x))."""
    return ft.reduce(lambda f, g: lambda x: g(f(x)), funcs, lambda x: x)

normalize = compose(str.strip, str.lower, lambda s: s.replace("--", " "), str.split)
print(normalize("  The Fulton County Grand Jury -- said Friday  "))

# reduce() to chain DataFrame merges on a shared key
parts = [titanic.groupby("pclass")[c].mean().rename(c).reset_index() for c in ["age", "fare", "survived"]]
print(ft.reduce(lambda l, r: l.merge(r, on="pclass"), parts).round(2))

['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday']
   pclass    age   fare  survived
0       1  38.23  84.15      0.63
1       2  29.88  20.66      0.47
2       3  25.14  13.68      0.24


### 5.3 `lru_cache()` / `cache`
Memoriza resultados de chamadas com os mesmos argumentos (que precisam ser **hasháveis**).
No Dia 11 vi `lru_cache` como exemplo de decorator; aqui o foco é **medir o ganho** em um caso
de NLP: calcular métricas por palavra repetidas milhares de vezes num corpus.

In [23]:
VOWELS = set("aeiouy")

def count_syllables(word: str) -> int:
    """Approximate English syllable count with a vowel-group heuristic."""
    word = word.lower()
    groups = sum(1 for k, _ in it.groupby(word, key=lambda ch: ch in VOWELS) if k)
    if word.endswith("e") and groups > 1 and not word.endswith("le"):
        groups -= 1
    return max(groups, 1)

cached_syllables = ft.lru_cache(maxsize=None)(count_syllables)

tokens = list(read_lowercase_tokens("learned"))
t_plain = timeit.timeit(lambda: [count_syllables(w) for w in tokens], number=1)
t_cache = timeit.timeit(lambda: [cached_syllables(w) for w in tokens], number=1)
print(f"tokens={len(tokens):,} | no cache={t_plain:.3f}s | lru_cache={t_cache:.3f}s")
print(cached_syllables.cache_info())
print({w: cached_syllables(w) for w in ["education", "language", "data", "the", "readable"]})

tokens=157,035 | no cache=0.197s | lru_cache=0.032s
CacheInfo(hits=143525, misses=13510, maxsize=None, currsize=13510)
{'education': 4, 'language': 2, 'data': 2, 'the': 1, 'readable': 3}


In [24]:
@ft.lru_cache(maxsize=None)
def edit_distance(a: str, b: str) -> int:
    """Levenshtein distance computed recursively with memoization."""
    if not a:
        return len(b)
    if not b:
        return len(a)
    cost = 0 if a[-1] == b[-1] else 1
    return min(
        edit_distance(a[:-1], b) + 1,
        edit_distance(a, b[:-1]) + 1,
        edit_distance(a[:-1], b[:-1]) + cost,
    )

# Typical learner misspellings vs. target words
for wrong, right in [("recieve", "receive"), ("goverment", "government"), ("beatiful", "beautiful")]:
    print(f"{wrong:>10} -> {right:<10} distance={edit_distance(wrong, right)}")
print(edit_distance.cache_info())

   recieve -> receive    distance=2
 goverment -> government distance=1
  beatiful -> beautiful  distance=1
CacheInfo(hits=374, misses=262, maxsize=None, currsize=262)


> 💡 **Nota de aprendizado:** sem cache, a recursão de Levenshtein é exponencial; com `lru_cache`
> ela vira programação dinâmica automática. Cuidado: `maxsize=None` cresce sem limite — em produção,
> defina um `maxsize` ou use `cache_clear()`.

### 5.4 `total_ordering`
Basta definir `__eq__` e **um** entre `__lt__`, `__le__`, `__gt__`, `__ge__`; o decorator gera os demais.
Útil para objetos de domínio ordenáveis — por exemplo, **níveis de proficiência**.

In [25]:
@ft.total_ordering
class ProficiencyBand:
    """An ordered vocabulary-frequency band, inspired by (not equal to) CEFR levels."""

    ORDER = ("A1", "A2", "B1", "B2", "C1", "C2")

    def __init__(self, label: str):
        if label not in self.ORDER:
            raise ValueError(f"unknown band: {label}")
        self.label = label

    def __eq__(self, other):
        if not isinstance(other, ProficiencyBand):
            return NotImplemented
        return self.label == other.label

    def __lt__(self, other):
        if not isinstance(other, ProficiencyBand):
            return NotImplemented
        return self.ORDER.index(self.label) < self.ORDER.index(other.label)

    def __hash__(self):
        return hash(self.label)

    def __repr__(self):
        return f"Band({self.label})"


b1, c1 = ProficiencyBand("B1"), ProficiencyBand("C1")
print(b1 < c1, b1 >= c1, c1 > b1, b1 <= b1)  # >=, >, <= were generated
print(sorted(map(ProficiencyBand, ["C2", "A1", "B2", "A2"])))
print(max(map(ProficiencyBand, ["B1", "A2", "C1"])))

True False True True
[Band(A1), Band(A2), Band(B2), Band(C2)]
Band(C1)


## 6. Aplicação prática: pipeline de processamento de texto educacional

**Objetivo:** estimar a **dificuldade lexical** de textos para aprendizes de inglês, combinando
`itertools` e `functools`.

**Honestidade metodológica:** não há uma lista CEFR oficial embutida no NLTK. Por isso construo
**faixas de frequência** (A1–C2) a partir do *ranking* de frequência do Brown Corpus inteiro — um
**proxy** amplamente usado em pesquisa de legibilidade (palavras frequentes tendem a ser aprendidas
antes), mas **não** equivalente ao English Vocabulary Profile/CEFR oficial.

Etapas:
1. Contar frequências no corpus (`chain.from_iterable` + `Counter`).
2. Atribuir faixas por ranking (`accumulate` + `islice`).
3. Classificar cada token com função memorizada (`lru_cache`) e configurada (`partial`).
4. Agregar por gênero (`groupby`) e resumir (`reduce`).

In [26]:
# Step 1: global frequency list from the whole Brown corpus (lazy stream into a Counter)
corpus_stream = (w.lower() for w in brown.words() if w.isalpha())
freq = Counter(corpus_stream)
ranked_words = [w for w, _ in freq.most_common()]
print(f"types={len(ranked_words):,} | top 15: {ranked_words[:15]}")

types=40,234 | top 15: ['the', 'of', 'and', 'to', 'a', 'in', 'that', 'is', 'was', 'he', 'for', 'it', 'with', 'as', 'his']


In [27]:
# Step 2: frequency bands by rank — band sizes chosen to mimic cumulative vocabulary targets
BAND_SIZES = {"A1": 500, "A2": 500, "B1": 1000, "B2": 2000, "C1": 4000, "C2": 8000}
band_edges = list(it.accumulate(BAND_SIZES.values()))
print("cumulative band edges:", dict(zip(BAND_SIZES, band_edges)))

word_band = {}
starts = [0, *band_edges[:-1]]
for band, start, stop_ in zip(BAND_SIZES, starts, band_edges):
    for w in it.islice(ranked_words, start, stop_):
        word_band[w] = band

coverage = sum(freq[w] for w in word_band) / sum(freq.values())
print(f"words with a band: {len(word_band):,} | token coverage: {coverage:.1%}")

cumulative band edges: {'A1': 500, 'A2': 1000, 'B1': 2000, 'B2': 4000, 'C1': 8000, 'C2': 16000}
words with a band: 16,000 | token coverage: 96.3%


In [28]:
# Step 3: memoized, configurable classifier
@ft.lru_cache(maxsize=None)
def classify_word(word: str, default: str = "OFF") -> str:
    """Return the frequency band of a word, or `default` if outside the band list."""
    return word_band.get(word, default)

def lexical_profile(tokens, stopword_set=frozenset(), include_off=True):
    """Return the proportion of tokens in each band for a token sequence."""
    words = (t.lower() for t in tokens if t.isalpha() and t.lower() not in stopword_set)
    bands = Counter(map(classify_word, words))
    if not include_off:
        bands.pop("OFF", None)
    n = sum(bands.values()) or 1
    return {b: round(bands.get(b, 0) / n, 3) for b in (*BAND_SIZES, "OFF") if include_off or b != "OFF"}

content_profile = ft.partial(lexical_profile, stopword_set=frozenset(stop))
print("news (content words):", content_profile(brown.words(categories="news")))

news (content words): {'A1': 0.279, 'A2': 0.139, 'B1': 0.15, 'B2': 0.14, 'C1': 0.126, 'C2': 0.1, 'OFF': 0.065}


In [29]:
# Step 4: profile every Brown genre and rank by share of advanced vocabulary (C1+C2+OFF)
ADVANCED = ("C1", "C2", "OFF")
rows = [
    {"genre": g, **content_profile(brown.words(categories=g))}
    for g in brown.categories()
]
profiles = pd.DataFrame(rows).set_index("genre")
profiles["advanced_share"] = profiles[list(ADVANCED)].sum(axis=1)
print(profiles.sort_values("advanced_share", ascending=False).round(3))
print(classify_word.cache_info())

                    A1     A2     B1     B2     C1     C2    OFF  \
genre                                                              
reviews          0.262  0.119  0.130  0.142  0.126  0.106  0.115   
humor            0.313  0.107  0.117  0.127  0.121  0.102  0.113   
adventure        0.332  0.113  0.127  0.129  0.124  0.098  0.076   
fiction          0.335  0.112  0.127  0.132  0.120  0.092  0.082   
news             0.279  0.139  0.150  0.140  0.126  0.100  0.065   
science_fiction  0.334  0.109  0.134  0.132  0.125  0.087  0.079   
belles_lettres   0.304  0.128  0.136  0.142  0.117  0.092  0.081   
hobbies          0.275  0.130  0.153  0.152  0.125  0.094  0.071   
lore             0.299  0.126  0.142  0.144  0.121  0.092  0.075   
learned          0.270  0.146  0.151  0.157  0.123  0.089  0.064   
editorial        0.311  0.144  0.141  0.136  0.119  0.083  0.066   
romance          0.363  0.117  0.124  0.129  0.116  0.084  0.067   
religion         0.327  0.136  0.143  0.134  0.1

In [30]:
# groupby + reduce: sentence-level difficulty within one genre, grouped by sentence length bucket
def sentence_difficulty(sent):
    """Share of content words outside the A1-B1 bands in one sentence."""
    prof = content_profile(sent)
    return ft.reduce(operator.add, (prof[b] for b in ("B2", *ADVANCED)), 0.0)

def length_bucket(sent):
    """Bucket sentences by token count in steps of 10 (0-9, 10-19, ...)."""
    return min(len(sent) // 10 * 10, 50)

sents = sorted(brown.sents(categories="learned")[:2000], key=length_bucket)
for bucket, group in it.groupby(sents, key=length_bucket):
    scores = [sentence_difficulty(s) for s in group]
    print(f"len {bucket:>2}+ tokens: n={len(scores):4d} | mean difficulty={np.mean(scores):.3f}")

len  0+ tokens: n= 255 | mean difficulty=0.475


len 10+ tokens: n= 627 | mean difficulty=0.474
len 20+ tokens: n= 597 | mean difficulty=0.487
len 30+ tokens: n= 339 | mean difficulty=0.503
len 40+ tokens: n= 123 | mean difficulty=0.484
len 50+ tokens: n=  59 | mean difficulty=0.471


In [31]:
def suggest_simpler_texts(category: str, k: int = 3, max_len: int = 25):
    """Return the k easiest short sentences of a genre (lowest difficulty score)."""
    candidates = it.filterfalse(lambda s: len(s) > max_len or len(s) < 6, brown.sents(categories=category))
    scored = ((sentence_difficulty(s), " ".join(s)) for s in it.islice(candidates, 3000))
    return sorted(scored)[:k]

for score, text in suggest_simpler_texts("adventure"):
    print(f"{score:.2f} | {text}")

0.00 | ) But he read on .
0.00 | A man could make a mark there .
0.00 | A thing he did not like doing , generally .


> 💡 **Nota de aprendizado (Educação + NLP):** o resultado contraria a intuição! *reviews* e *humor*
> lideram em vocabulário raro (nomes próprios, títulos, palavras expressivas), enquanto *government*
> fica em último em C1+C2+OFF — mas tem as **maiores** fatias de A2–B2. Hipótese provável: as faixas
> foram derivadas do **próprio** Brown, então jargão técnico repetido muitas vezes (ex.: termos
> administrativos) acaba "promovido" a faixas frequentes; já *mystery* e *romance* são de fato os mais
> concentrados em A1. Lição metodológica: a lista de referência deve vir de um corpus **independente**
> (ex.: English Vocabulary Profile) para evitar circularidade. Isso é um **perfil lexical**, não um
> nível de leitura validado com aprendizes.

## 7. Aplicação DS: análise de dataset real com pipelines funcionais

Vou reescrever uma pequena análise do **Titanic** e dos **Penguins** em estilo funcional:
funções pequenas e puras, configuradas com `partial` e encadeadas com `reduce`, sobre iteradores.

In [32]:
def add_family_size(df):
    """Add family_size = sibsp + parch + 1."""
    return df.assign(family_size=df["sibsp"] + df["parch"] + 1)

def drop_missing(df, cols):
    """Drop rows with missing values in the given columns."""
    return df.dropna(subset=list(cols))

def add_age_band(df, edges, labels):
    """Categorize age into bands."""
    return df.assign(age_band=pd.cut(df["age"], bins=edges, labels=labels))

def pipe(data, *steps):
    """Apply steps left-to-right with functools.reduce."""
    return ft.reduce(lambda acc, step: step(acc), steps, data)

clean_titanic = pipe(
    titanic,
    ft.partial(drop_missing, cols=["age", "embarked"]),
    add_family_size,
    ft.partial(add_age_band, edges=[0, 12, 18, 40, 60, 100], labels=["child", "teen", "adult", "middle", "senior"]),
)
print(clean_titanic.shape)
print(clean_titanic.pivot_table(index="age_band", columns="sex", values="survived", aggfunc="mean", observed=True).round(2))

(712, 17)
sex       female  male
age_band              
child       0.59  0.57
teen        0.75  0.09
adult       0.78  0.18
middle      0.76  0.19
senior      1.00  0.11


In [33]:
# product() over categorical levels: survival for every (class, sex) cell, including empty ones
levels = it.product(sorted(clean_titanic["pclass"].unique()), sorted(clean_titanic["sex"].unique()))
summary = []
for pclass, sex in levels:
    cell = clean_titanic.query("pclass == @pclass and sex == @sex")["survived"]
    summary.append((pclass, sex, len(cell), round(cell.mean(), 3)))
print(pd.DataFrame(summary, columns=["pclass", "sex", "n", "survival_rate"]))

   pclass     sex    n  survival_rate
0       1  female   83          0.964
1       1    male  101          0.396
2       2  female   74          0.919
3       2    male   99          0.152
4       3  female  102          0.461
5       3    male  253          0.150


In [34]:
# Penguins: groupby + accumulate to build cumulative mass distribution per species
for species, group in it.groupby(
    sorted(penguins[["species", "body_mass_g"]].itertuples(index=False), key=lambda r: r.species),
    key=lambda r: r.species,
):
    masses = sorted(r.body_mass_g for r in group)
    cum = list(it.accumulate(masses))
    half = next(i for i, c in enumerate(cum) if c >= cum[-1] / 2)
    print(f"{species:<10} n={len(masses):3d} | mass where 50% of total biomass is reached: {masses[half]:.0f} g")

Adelie     n=146 | mass where 50% of total biomass is reached: 3750 g
Chinstrap  n= 68 | mass where 50% of total biomass is reached: 3725 g
Gentoo     n=119 | mass where 50% of total biomass is reached: 5150 g


In [35]:
# Diabetes: rank feature pairs by joint correlation with the target using combinations + partial
target = diabetes.frame["target"]
corr_with_target = ft.partial(lambda col, y: diabetes.frame[col].corr(y), y=target)
single = {c: round(corr_with_target(c), 3) for c in diabetes.feature_names}
pair_scores = sorted(
    ((a, b, round(abs(single[a]) + abs(single[b]), 3)) for a, b in it.combinations(diabetes.feature_names, 2)),
    key=operator.itemgetter(2),
    reverse=True,
)
print("single-feature r:", single)
print("top pairs by summed |r|:", pair_scores[:3])

single-feature r: {'age': 0.188, 'sex': 0.043, 'bmi': 0.586, 'bp': 0.441, 's1': 0.212, 's2': 0.174, 's3': -0.395, 's4': 0.43, 's5': 0.566, 's6': 0.382}
top pairs by summed |r|: [('bmi', 's5', 1.152), ('bmi', 'bp', 1.027), ('bmi', 's4', 1.016)]


## 8. Comparação de performance: `itertools` vs list comprehensions

Duas dimensões:
- **Memória** (`sys.getsizeof`): um iterador ocupa tamanho **constante**; uma lista cresce com n.
- **Velocidade** (`timeit`): depende do caso. Para consumir tudo, listas podem ser tão rápidas
  quanto ou mais rápidas; a vantagem de `itertools` aparece quando **não** precisamos de tudo
  (curto-circuito) ou em achatamento/combinação implementados em C.

In [36]:
# Memory: list vs lazy iterator over the same Brown token stream
tokens_all = [w.lower() for w in brown.words() if w.isalpha()]
as_list = [t for t in tokens_all if len(t) > 3]
as_iter = filter(lambda t: len(t) > 3, tokens_all)
as_gen = (t for t in tokens_all if len(t) > 3)
as_chain = it.chain(tokens_all, tokens_all)

mem = pd.DataFrame(
    {
        "object": ["list comprehension", "filter() iterator", "generator expression", "chain() of 2 lists"],
        "bytes": [sys.getsizeof(o) for o in (as_list, as_iter, as_gen, as_chain)],
    }
)
print(f"tokens: {len(tokens_all):,}")
print(mem)

tokens: 981,716
                 object    bytes
0    list comprehension  4688312
1     filter() iterator       48
2  generator expression      216
3    chain() of 2 lists       48


In [37]:
# Speed 1: flattening nested lists — chain.from_iterable vs nested comprehension vs sum()
nested = [list(s) for s in brown.sents(categories="news")]
n = 20
results = {
    "chain.from_iterable": timeit.timeit(lambda: list(it.chain.from_iterable(nested)), number=n),
    "nested comprehension": timeit.timeit(lambda: [t for s in nested for t in s], number=n),
    "reduce(operator.iconcat)": timeit.timeit(lambda: ft.reduce(operator.iconcat, nested, []), number=n),
}
for k, v in sorted(results.items(), key=operator.itemgetter(1)):
    print(f"{k:<26} {v / n * 1000:7.2f} ms per run")

reduce(operator.iconcat)      0.67 ms per run
chain.from_iterable           0.94 ms per run
nested comprehension          1.31 ms per run


In [38]:
# Speed 2: short-circuit — first 1,000 long words: islice over generator vs full list then slice
n = 20
t_islice = timeit.timeit(lambda: list(it.islice((t for t in tokens_all if len(t) > 8), 1000)), number=n)
t_list = timeit.timeit(lambda: [t for t in tokens_all if len(t) > 8][:1000], number=n)
print(f"islice (lazy)        : {t_islice / n * 1000:8.3f} ms")
print(f"list then slice      : {t_list / n * 1000:8.3f} ms")
print(f"speed-up             : {t_list / t_islice:8.1f}x")

islice (lazy)        :    0.178 ms
list then slice      :   21.605 ms
speed-up             :    121.1x


In [39]:
# Speed 3: cumulative sum — accumulate vs manual loop vs numpy.cumsum
fares_all = titanic["fare"].tolist() * 200
def manual_cumsum(xs):
    """Running sum with an explicit Python loop."""
    out, total = [], 0.0
    for x in xs:
        total += x
        out.append(total)
    return out

n = 20
arr = np.array(fares_all)
for name, fn in {
    "itertools.accumulate": lambda: list(it.accumulate(fares_all)),
    "manual loop": lambda: manual_cumsum(fares_all),
    "numpy.cumsum": lambda: np.cumsum(arr),
}.items():
    print(f"{name:<22} {timeit.timeit(fn, number=n) / n * 1000:7.3f} ms  (n={len(fares_all):,})")

itertools.accumulate     4.041 ms  (n=178,200)
manual loop              4.401 ms  (n=178,200)
numpy.cumsum             0.429 ms  (n=178,200)


> 💡 **Nota de aprendizado:** os números variam por máquina, mas o padrão costuma ser:
> (1) iteradores têm **memória constante**; (2) `islice` com geradores vence quando só preciso de
> parte dos dados; (3) para achatar listas, `chain.from_iterable` costuma superar a compreensão
> aninhada, mas `reduce(operator.iconcat)` (que estende uma única lista in-place) pode ser ainda mais
> rápido; (4) `accumulate` tende a superar o laço manual, mas para dados **numéricos vetorizados**
> o NumPy continua imbatível. Regra prática: *NumPy/pandas para números, itertools para streams
> heterogêneos e texto.*

## 9. Exercícios práticos com soluções

### Exercício 1 — Bigramas com `itertools`
Escreva `bigrams(tokens)` usando **apenas** `itertools` (sem índices) e liste os 10 bigramas de
palavras de conteúdo mais frequentes do gênero *news*. *(Dica: `it.pairwise`, Python 3.10+, ou `tee` + `islice`.)*

In [40]:
# Solution 1
def bigrams(tokens):
    """Yield consecutive token pairs lazily using tee + islice (works on any iterable)."""
    a, b = it.tee(tokens)
    return zip(a, it.islice(b, 1, None))

news_content = (w.lower() for w in brown.words(categories="news") if w.isalpha() and w.lower() not in stop)
print(Counter(bigrams(news_content)).most_common(10))
assert list(bigrams("abc")) == list(it.pairwise("abc")) == [("a", "b"), ("b", "c")]

[(('new', 'york'), 52), (('per', 'cent'), 50), (('united', 'states'), 37), (('last', 'week'), 35), (('last', 'year'), 34), (('white', 'house'), 29), (('high', 'school'), 23), (('home', 'runs'), 23), (('president', 'kennedy'), 20), (('last', 'night'), 18)]


### Exercício 2 — Lotes (*batches*) para processamento
Implemente `batched(iterable, n)` com `islice` (equivalente ao `itertools.batched` do Python 3.12)
e use para calcular a média de `fare` do Titanic em lotes de 100 passageiros.

In [41]:
# Solution 2
def batched(iterable, n):
    """Yield tuples of up to n items from iterable (last batch may be shorter)."""
    if n < 1:
        raise ValueError("n must be >= 1")
    iterator = iter(iterable)
    while batch := tuple(it.islice(iterator, n)):
        yield batch

batch_means = [round(np.mean(b), 2) for b in batched(titanic["fare"], 100)]
print(batch_means)
assert sum(len(b) for b in batched(range(891), 100)) == 891
assert list(batched([], 3)) == []

[29.52, 25.52, 34.1, 43.83, 25.94, 32.16, 34.47, 38.77, 24.87]


### Exercício 3 — Escada de palavras com `lru_cache`
Usando `edit_distance` (seção 5.3) e `partial`, encontre no vocabulário das faixas **A1–A2**
as 5 palavras mais próximas de cada erro de aprendiz: `"freind"`, `"wich"`, `"untill"`.

In [42]:
# Solution 3
basic_vocab = [w for w, b in word_band.items() if b in ("A1", "A2")]

def nearest_words(misspelling, vocab, k=5):
    """Return the k vocabulary words with the smallest edit distance to the misspelling."""
    distance_to = ft.partial(edit_distance, misspelling)
    return sorted(vocab, key=lambda w: (distance_to(w), w))[:k]

for wrong in ["freind", "wich", "untill"]:
    print(f"{wrong:>7} -> {nearest_words(wrong, basic_vocab)}")

 freind -> ['find', 'friend', 'around', 'behind', 'being']
   wich -> ['which', 'wish', 'with', 'each', 'high']
 untill -> ['until', 'still', 'bill', 'entire', 'tell']


### Exercício 4 — Ordenação com `total_ordering` + `groupby`
Crie a classe `ScoredSentence` (texto + dificuldade) com `@total_ordering`, ordene 500 frases do
gênero *government* e agrupe-as em faixas de dificuldade de 0.1 com `groupby`.

In [43]:
# Solution 4
@ft.total_ordering
class ScoredSentence:
    """A sentence with a lexical difficulty score; ordered by score."""

    def __init__(self, tokens):
        self.text = " ".join(tokens)
        self.score = sentence_difficulty(tokens)

    def __eq__(self, other):
        return self.score == other.score

    def __lt__(self, other):
        return self.score < other.score

    def __repr__(self):
        return f"ScoredSentence({self.score:.2f}, {self.text[:40]!r})"

scored = sorted(map(ScoredSentence, brown.sents(categories="government")[:500]))
band_of = lambda s: round(min(s.score, 0.99) // 0.1 * 0.1, 1)
print({band: len(list(g)) for band, g in it.groupby(scored, key=band_of)})
print("easiest:", scored[0])
print("hardest:", scored[-1])

{0.0: 72, 0.1: 35, 0.2: 82, 0.3: 90, 0.4: 130, 0.5: 22, 0.6: 23, 0.7: 9, 0.8: 1, 0.9: 36}
easiest: ScoredSentence(0.00, 'Contact')
hardest: ScoredSentence(1.00, 'Purchasing practices .')


### Exercício 5 — Pipeline completo em uma linha (desafio)
Usando `compose` (seção 5.2), `chain.from_iterable`, `filterfalse` e `Counter`, gere as 10 palavras
de conteúdo mais frequentes nos gêneros *hobbies* e *lore* juntos.

In [44]:
# Solution 5
top_content = compose(
    lambda cats: it.chain.from_iterable(brown.words(categories=c) for c in cats),
    lambda toks: (t.lower() for t in toks if t.isalpha()),
    lambda toks: it.filterfalse(stop.__contains__, toks),
    Counter,
    lambda c: c.most_common(10),
)
print(top_content(["hobbies", "lore"]))

[('one', 647), ('may', 313), ('time', 311), ('new', 311), ('first', 287), ('would', 271), ('many', 262), ('two', 250), ('also', 209), ('could', 201)]


## Resumo do Dia 12

| Ferramenta | Quando usar |
|---|---|
| `count`, `cycle`, `repeat` | IDs, *folds* round-robin, argumentos constantes |
| `chain`, `islice` | juntar e fatiar streams sem materializar listas |
| `takewhile`, `dropwhile` | cortar sequências ordenadas por limiar |
| `accumulate` | totais corridos, curvas de vocabulário |
| `zip_longest` | alinhar sequências de tamanhos diferentes |
| `groupby` | agrupar elementos consecutivos (**ordene antes!**) |
| `product`, `combinations`, `permutations` | grids de hiperparâmetros, pares de features, coocorrência |
| `partial` | pré-configurar funções de limpeza/transformação |
| `reduce` | combinar Counters, compor funções, encadear merges |
| `lru_cache` | memorizar métricas por palavra, recursões (Levenshtein) |
| `total_ordering` | objetos de domínio ordenáveis (faixas de proficiência) |

**Limitações:** as faixas A1–C2 são um **proxy de frequência** derivado do Brown Corpus (inglês
americano escrito de 1961), não a classificação CEFR oficial; os tempos de `timeit` dependem da máquina.

**Próximos passos:** aplicar esses padrões a corpora maiores com leitura em *streaming* e combinar
com `multiprocessing`/`concurrent.futures` para paralelizar etapas do pipeline.